In [1]:
using CMPSExcitations

In [55]:
# canonical basis
function projection_matrix1(D, R)
    Dr, M = eigen(R)

    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix2(D, R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    Dr, M = eigen(R)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        dD1 = view(e, 1:D)
        dD2 = view(e, D+1:2*D)
        X = zeros(D, D)

        k = 2D + 1
        for i in 1:D, j in 1:D
            if i != j
                X[i, j] = e[k] # sets the one hot vector
                k += 1
            end
        end

        Dr = Diagonal(Dr)
        W1 = M * ((X * Dr - Dr * X) + Diagonal(dD1)) / M
        W2 = M * ((X * Dr - Dr * X) + Diagonal(dD2)) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

function projection_matrix3(D, R)
    Dr, M = eigen(R)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E - M * Diagonal(F) / M
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

excitation_matrix (generic function with 1 method)

In [56]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hsingle(c, μ) = ∫(2 * ∂ψ̂' * ∂ψ̂ - 2 * μ * ψ̂' * ψ̂ + 4 * c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));

In [57]:
c, μ = 10., 5.
tol = 1e-10

Ds = [4]
D = maximum(Ds)

HLL = Hsingle(c, μ)
@time stateLL = find_groundstate(Ds, HLL, YangGaudinCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])

# -----
stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], stateLL.Rs[1]));
HCLL = Hcoupled(c, μ)
println("Energy density: ", expval(HCLL.h, stateCLL)[], "\nParticle density: ", expval(ψ̂₁' * ψ̂₁ + ψ̂₂' * ψ̂₂, stateCLL)[], "\nDensity imbalance: ", expval(ψ̂₁' * ψ̂₁ - ψ̂₂' * ψ̂₂, stateCLL)[])

Optimizing D=4


┌ Info: YangGaudinCMPS ground state: initialization with e = 5752.505004714906
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:106
┌ Info: LBFGS: converged after 272 iterations: f = -2.734747817523, ‖∇f‖ = 2.9376e-12
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 4 | YangGaudinCMPS{Constant{Matrix{Float64}}, 1}
  0.218214 seconds (1.30 M allocations: 60.795 MiB, 1.75% gc time)
---------------
  0.218313 seconds (1.30 M allocations: 60.799 MiB, 1.75% gc time)
Energy density: -2.734747817523249
 Particle density: 0.43244359486983464
 Order parameter: 0.4386898377587201
Energy density: -2.734747817523244
Particle density: 0.8648871897396727
Density imbalance: 0.0


┌ Info: YangGaudinCMPS ground state: converged after 273 iterations: e = -2.734747817523, ‖∇e‖ = 2.9376e-12
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/yanggaudin/groundstate.jl:118


In [58]:
function leftcanonical(state, cmps=false)
    leftgauge!(state)
    r = rightenv(state)[1][]
    D, U = eigen(r)
    Q = U \ state.Q[] * U
    R = U \ state.Rs[1][] * U
    return (cmps) ? InfiniteCMPS(Constant(Q), (Constant(R), Constant(R))) : (Q, R)
end

leftcanonical (generic function with 2 methods)

In [59]:
# common setup
stateCLL = leftcanonical(stateCLL, true)
p = 0 # momentum
R = stateCLL.Rs[1][]
D = size(R, 1) # R = MDᵣ/M
space = InfiniteCMPSExcitationSpace(p, stateCLL, stateCLL)
H = excitation_matrix(excitation_operator(HCLL, space), D);

In [63]:
P = projection_matrix1(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))


[0.5394709302676697, 3.348946770286046, 6.88106771075532, 8.988576699717235, 9.547054307783645]


In [62]:
P = Matrix(qr(projection_matrix1(D, R)).Q)
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[0.53947093026763, 3.348946770286063, 6.881067710755368, 8.988576699717205, 9.547054307783604]


In [64]:
P = projection_matrix2(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[0.5394709303147427, 3.3489467702955285, 6.8810677107761515, 8.988576699753818, 9.547054307786663]


In [65]:
P = Matrix(qr(projection_matrix2(D, R)).Q)
@assert P' * P ≈ I
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[0.5394709302676161, 3.3489467702860978, 6.881067710755407, 8.988576699717218, 9.547054307783592]


In [53]:
P = projection_matrix3(D, R)
vals, vecs = eigen(P' * H * P, P' * P)
println(real.(vals[1:5]))

[-0.9209521434177811, 0.23727140121030027, 1.3856716913044136, 2.179689582653853, 2.7206267549491656]


In [54]:
P = Matrix(qr(projection_matrix3(D, R)).Q)
@assert P' * P ≈ I
vals, vecs = eigen(P' * H * P)
println(real.(vals[1:5]))

[-0.920952143417699, 0.23727140121019835, 1.3856716913045395, 2.179689582653992, 2.7206267549491274]


QR instead of geneigsolve seems to be consistently equivalent as expected. Different parametrization seem to agree as well. However, the same parametrization gives different results based on the ground state which presumably only varies by gauge.. ??? Specifically the projector seems to be the issue.